## **Bell Pepper Disease Prediction System**

**Developer:** Sanusi Shafii

**Email:** s.shafii27683@fudutsinma.edu.com



### Setup and Imports

In [ ]:
# Install required packages
#!pip install -q tensorflow opencv-python matplotlib seaborn pandas numpy scikit-learn pillow

# Import libraries
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.utils import plot_model
import cv2
from google.colab import files
import zipfile
import ipywidgets as widgets
from IPython.display import display, clear_output, Image as IPImage
import io
from PIL import Image
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)

# Check TensorFlow version and GPU
print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")

: 

Set Local Dataset Path

In [ ]:
# Mount Google Drive to access dataset
from google.colab import drive
drive.mount('/content/drive')

# Set dataset path
DATASET_PATH = '/content/drive/MyDrive/Colab Notebooks/ml2-main/dataset'
print(f"Dataset path: {DATASET_PATH}")

# Check if dataset exists
if not os.path.exists(DATASET_PATH):
    print(f"ERROR: Dataset path '{DATASET_PATH}' does not exist!")
    print("Please update the DATASET_PATH variable to point to your dataset directory.")
else:
    print("Dataset path exists.")

    # List classes (subdirectories)
    classes = [d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))]
    print(f"Found {len(classes)} class(es): {classes}")

    # Count images per class
    for cls in classes:
        cls_path = os.path.join(DATASET_PATH, cls)
        num_images = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
        print(f"  {cls}: {num_images} images")

Data Exploration and Visualization

In [ ]:
def explore_dataset(dataset_path):
    """Explore and visualize the dataset"""

    # Get classes
    classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])

    if not classes:
        print("No classes found. Check dataset structure.")
        return

    print("=" * 60)
    print("DATASET EXPLORATION")
    print("=" * 60)
    print(f"Number of classes: {len(classes)}")
    print(f"Classes: {classes}\n")

    # Collect statistics
    class_stats = []
    total_images = 0

    for cls in classes:
        cls_path = os.path.join(dataset_path, cls)
        image_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]
        count = len(image_files)
        class_stats.append((cls, count))
        total_images += count

        # Display first few image names
        if image_files:
            print(f"{cls} ({count} images):")
            print(f"  Sample files: {', '.join(image_files[:3])}{'...' if len(image_files) > 3 else ''}")

    print(f"\nTotal images in dataset: {total_images}")

    # Visualize class distribution
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Bar chart
    class_names = [stat[0] for stat in class_stats]
    class_counts = [stat[1] for stat in class_stats]

    bars = axes[0].bar(class_names, class_counts, color=['green', 'red', 'blue', 'orange'][:len(class_names)])
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Number of Images')
    axes[0].set_title('Class Distribution')
    axes[0].tick_params(axis='x', rotation=45)

    # Add count labels on bars
    for bar, count in zip(bars, class_counts):
        height = bar.get_height()
        axes[0].text(bar.get_x() + bar.get_width()/2., height + 0.5,
                    f'{count}', ha='center', va='bottom')

    # Pie chart
    if len(class_stats) > 1:
        axes[1].pie(class_counts, labels=class_names, autopct='%1.1f%%',
                   colors=['green', 'red', 'blue', 'orange'][:len(class_names)])
        axes[1].set_title('Class Proportion')
    else:
        axes[1].text(0.5, 0.5, 'Single class dataset\nCannot create pie chart',
                    ha='center', va='center', fontsize=12)
        axes[1].set_title('Class Proportion')

    plt.tight_layout()
    plt.show()

    return classes

# Run exploration
classes = explore_dataset(DATASET_PATH)

Display Sample Images

In [ ]:
def display_sample_images(dataset_path, num_samples=3):
    """Display sample images from each class"""

    classes = sorted([d for d in os.listdir(dataset_path) if os.path.isdir(os.path.join(dataset_path, d))])

    if not classes:
        print("No classes found to display.")
        return

    fig, axes = plt.subplots(len(classes), num_samples, figsize=(15, 5 * len(classes)))

    if len(classes) == 1:
        axes = np.array([axes]).reshape(1, -1)

    for row_idx, cls in enumerate(classes):
        cls_path = os.path.join(dataset_path, cls)
        image_files = [f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))][:num_samples]

        for col_idx, img_file in enumerate(image_files):
            if col_idx >= num_samples:
                break

            img_path = os.path.join(cls_path, img_file)
            try:
                img = cv2.imread(img_path)
                if img is None:
                    print(f"Warning: Could not read image {img_path}")
                    continue

                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                axes[row_idx, col_idx].imshow(img)
                axes[row_idx, col_idx].set_title(f"{cls}\n{img_file}", fontsize=10)
                axes[row_idx, col_idx].axis('off')

            except Exception as e:
                print(f"Error loading image {img_path}: {e}")
                axes[row_idx, col_idx].text(0.5, 0.5, f"Error\n{img_file}",
                                          ha='center', va='center')
                axes[row_idx, col_idx].axis('off')

    # Hide any unused subplots
    for i in range(len(classes)):
        for j in range(len(image_files), num_samples):
            axes[i, j].axis('off')

    plt.suptitle(f'Sample Images from Dataset', fontsize=16, y=1.02)
    plt.tight_layout()
    plt.show()

# Display samples
display_sample_images(DATASET_PATH, num_samples=3)

 Data Preprocessing and Augmentation

In [ ]:
# Set image parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 5  # Reduced epochs as requested

print("DATA PREPROCESSING SETUP")
print("=" * 40)
print(f"Image dimensions: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Training epochs: {EPOCHS}")

# Check if dataset has enough images
classes = [d for d in os.listdir(DATASET_PATH) if os.path.isdir(os.path.join(DATASET_PATH, d))]
for cls in classes:
    cls_path = os.path.join(DATASET_PATH, cls)
    num_images = len([f for f in os.listdir(cls_path) if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))])
    print(f"{cls}: {num_images} images")

# Create data generators
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2  # 80% train, 20% validation
)

val_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=0.2
)

# Create data generators
train_generator = train_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

val_generator = val_datagen.flow_from_directory(
    DATASET_PATH,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

# Get class information
class_names = list(train_generator.class_indices.keys())
num_classes = len(class_names)

print("\nCLASS INFORMATION")
print("=" * 40)
print(f"Number of classes: {num_classes}")
print(f"Classes: {class_names}")
print(f"Class indices: {train_generator.class_indices}")
print(f"Training samples: {train_generator.samples}")
print(f"Validation samples: {val_generator.samples}")

# Calculate steps per epoch
train_steps = max(1, train_generator.samples // BATCH_SIZE)
val_steps = max(1, val_generator.samples // BATCH_SIZE)
print(f"Training steps per epoch: {train_steps}")
print(f"Validation steps per epoch: {val_steps}")

Visualize Augmented Images

In [ ]:
def visualize_augmentation(generator, class_names, num_images=8):
    """Visualize augmented training images"""

    # Get a batch of images
    images, labels = next(generator)

    # Create figure
    plt.figure(figsize=(15, 10))

    # Display images
    for i in range(min(num_images, len(images))):
        plt.subplot(3, 4, i + 1)
        plt.imshow(images[i])

        # Get class label
        label_idx = np.argmax(labels[i])
        class_name = class_names[label_idx]

        plt.title(f"Class: {class_name}", fontsize=11)
        plt.axis('off')

    plt.suptitle('Augmented Training Images (Sample)', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()

# Visualize augmented images
print("Visualizing augmented images...")
visualize_augmentation(train_generator, class_names)

 Build Neural Network Model

In [ ]:
def create_model(num_classes, img_height=224, img_width=224):
    """Create a MobileNetV2-based model for disease classification"""

    # Load pre-trained MobileNetV2
    base_model = MobileNetV2(
        input_shape=(img_height, img_width, 3),
        include_top=False,
        weights='imagenet',
        pooling='avg'
    )

    # Freeze base model
    base_model.trainable = False

    # Create model
    inputs = keras.Input(shape=(img_height, img_width, 3))
    x = base_model(inputs, training=False)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.5)(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    model = keras.Model(inputs=inputs, outputs=outputs)

    # Compile model
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0001),
        loss='categorical_crossentropy',
        metrics=['accuracy',
                keras.metrics.Precision(name='precision'),
                keras.metrics.Recall(name='recall')]
    )

    return model, base_model

# Create model
print("Building neural network model...")
model, base_model = create_model(num_classes)

# Display model summary
print("\nMODEL ARCHITECTURE")
print("=" * 40)
model.summary()

Visualize Model Architecture

Define Training Callbacks

In [ ]:
# Define training callbacks
print("Setting up training callbacks...")

# Early stopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True,
    verbose=1,
    mode='min'
)

# Reduce learning rate on plateau
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=3,
    min_lr=1e-7,
    verbose=1,
    mode='min'
)

# Model checkpoint
checkpoint = ModelCheckpoint(
    'best_bell_pepper_model.h5',
    monitor='val_accuracy',
    save_best_only=True,
    mode='max',
    verbose=1
)

# Callbacks list
callbacks = [early_stopping, reduce_lr, checkpoint]

print("\nTRAINING CALLBACKS")
print("=" * 40)
for i, callback in enumerate(callbacks, 1):
    print(f"{i}. {callback.__class__.__name__}")
    if hasattr(callback, 'monitor'):
        print(f"   - Monitor: {callback.monitor}")
    if hasattr(callback, 'patience'):
        print(f"   - Patience: {callback.patience}")

Train the Model

In [ ]:
# Train the model
print("Starting model training...")
print(f"Training for {EPOCHS} epochs")
print("=" * 60)

# Train model
history = model.fit(
    train_generator,
    steps_per_epoch=train_steps,
    epochs=EPOCHS,
    validation_data=val_generator,
    validation_steps=val_steps,
    callbacks=callbacks,
    verbose=1
)

# Save final model
model.save('bell_pepper_disease_model_final.h5')
print("\n" + "=" * 60)
print("Training completed successfully!")
print(f"Model saved as: bell_pepper_disease_model_final.h5")
print("=" * 60)

 Plot Training History

In [ ]:
def plot_training_history(history):
    """Plot training metrics"""

    # Create figure
    fig, axes = plt.subplots(2, 2, figsize=(15, 10))

    # Accuracy plot
    axes[0, 0].plot(history.history['accuracy'], 'b-', linewidth=2, label='Training')
    axes[0, 0].plot(history.history['val_accuracy'], 'r-', linewidth=2, label='Validation')
    axes[0, 0].set_title('Model Accuracy', fontsize=14)
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Accuracy')
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Loss plot
    axes[0, 1].plot(history.history['loss'], 'b-', linewidth=2, label='Training')
    axes[0, 1].plot(history.history['val_loss'], 'r-', linewidth=2, label='Validation')
    axes[0, 1].set_title('Model Loss', fontsize=14)
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('Loss')
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Precision plot (if available)
    if 'precision' in history.history:
        axes[1, 0].plot(history.history['precision'], 'b-', linewidth=2, label='Training')
        axes[1, 0].plot(history.history['val_precision'], 'r-', linewidth=2, label='Validation')
        axes[1, 0].set_title('Model Precision', fontsize=14)
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Precision')
        axes[1, 0].legend()
        axes[1, 0].grid(True, alpha=0.3)

    # Recall plot (if available)
    if 'recall' in history.history:
        axes[1, 1].plot(history.history['recall'], 'b-', linewidth=2, label='Training')
        axes[1, 1].plot(history.history['val_recall'], 'r-', linewidth=2, label='Validation')
        axes[1, 1].set_title('Model Recall', fontsize=14)
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Recall')
        axes[1, 1].legend()
        axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle('Training History', fontsize=16)
    plt.tight_layout()
    plt.show()

    # Print final metrics
    print("\nFINAL TRAINING METRICS")
    print("=" * 40)
    final_epoch = len(history.history['accuracy']) - 1
    print(f"Epoch {final_epoch + 1} Results:")
    print(f"  Training Accuracy:   {history.history['accuracy'][-1]:.4f} ({history.history['accuracy'][-1]*100:.2f}%)")
    print(f"  Validation Accuracy: {history.history['val_accuracy'][-1]:.4f} ({history.history['val_accuracy'][-1]*100:.2f}%)")
    print(f"  Training Loss:       {history.history['loss'][-1]:.4f}")
    print(f"  Validation Loss:     {history.history['val_loss'][-1]:.4f}")

# Plot training history
plot_training_history(history)

 Model Evaluation

In [ ]:
def evaluate_model(model, val_generator, class_names):
    """Evaluate model performance"""

    print("MODEL EVALUATION")
    print("=" * 60)

    # Reset generator
    val_generator.reset()

    # Get predictions
    predictions = model.predict(val_generator, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = val_generator.classes

    # Calculate accuracy
    accuracy = np.sum(y_pred == y_true) / len(y_true)

    # Classification report
    print("\nCLASSIFICATION REPORT")
    print("-" * 40)
    print(classification_report(y_true, y_pred, target_names=class_names, digits=4))

    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=class_names, yticklabels=class_names,
                square=True, cbar_kws={"shrink": 0.8})

    plt.title(f'Confusion Matrix\nOverall Accuracy: {accuracy*100:.2f}%', fontsize=14)
    plt.ylabel('Actual Class', fontsize=12)
    plt.xlabel('Predicted Class', fontsize=12)
    plt.tight_layout()
    plt.show()

    # Per-class accuracy
    print("\nPER-CLASS ACCURACY")
    print("-" * 40)
    for i, class_name in enumerate(class_names):
        class_mask = y_true == i
        if np.sum(class_mask) > 0:
            class_acc = np.sum(y_pred[class_mask] == y_true[class_mask]) / np.sum(class_mask)
            print(f"{class_name:15s}: {class_acc*100:6.2f}% ({np.sum(class_mask)} samples)")

    return accuracy

# Evaluate model
val_accuracy = evaluate_model(model, val_generator, class_names)

 Create Prediction Class

In [ ]:
class BellPepperDiseaseDetector:
    """Class for detecting diseases in bell pepper leaves"""

    def __init__(self, model_path=None, class_names=None):
        # Load model
        if model_path and os.path.exists(model_path):
            self.model = keras.models.load_model(model_path)
        else:
            self.model = model  # Use the trained model

        # Set class names
        if class_names:
            self.class_names = class_names
        else:
            self.class_names = ['Healthy', 'Diseased']  # Default

        # Image parameters
        self.img_height = 224
        self.img_width = 224

    def preprocess_image(self, image):
        """Preprocess image for prediction"""
        # Convert PIL Image to numpy array if needed
        if isinstance(image, Image.Image):
            image = np.array(image)

        # Ensure image has 3 channels
        if len(image.shape) == 2:
            image = cv2.cvtColor(image, cv2.COLOR_GRAY2RGB)
        elif image.shape[2] == 4:
            image = cv2.cvtColor(image, cv2.COLOR_RGBA2RGB)

        # Store original for display
        original = image.copy()

        # Resize and normalize
        image = cv2.resize(image, (self.img_width, self.img_height))
        image = image / 255.0
        image = np.expand_dims(image, axis=0)

        return image, original

    def predict(self, image):
        """Make prediction on image"""
        try:
            # Preprocess
            img_array, original_img = self.preprocess_image(image)

            # Predict
            predictions = self.model.predict(img_array, verbose=0)[0]

            # Get results
            pred_idx = np.argmax(predictions)
            pred_class = self.class_names[pred_idx]
            confidence = predictions[pred_idx] * 100

            # All probabilities
            all_probs = predictions

            return {
                'predicted_class': pred_class,
                'confidence': confidence,
                'all_probabilities': all_probs,
                'original_image': original_img,
                'class_index': pred_idx
            }

        except Exception as e:
            print(f"Prediction error: {e}")
            return None

    def display_results(self, result):
        """Display prediction results"""
        if not result:
            print("No results to display")
            return

        # Create figure
        fig, axes = plt.subplots(1, 2, figsize=(14, 6))

        # Show original image
        axes[0].imshow(result['original_image'])
        axes[0].set_title('Uploaded Leaf Image', fontsize=14)
        axes[0].axis('off')

        # Show probabilities
        x_pos = np.arange(len(self.class_names))
        colors = ['green' if 'healthy' in cls.lower() else 'red' for cls in self.class_names]

        bars = axes[1].barh(x_pos, result['all_probabilities'] * 100, color=colors)
        axes[1].set_yticks(x_pos)
        axes[1].set_yticklabels(self.class_names)
        axes[1].set_xlabel('Confidence (%)', fontsize=12)
        axes[1].set_xlim([0, 100])
        axes[1].set_title('Disease Prediction', fontsize=14)

        # Add confidence values on bars
        for i, (bar, prob) in enumerate(zip(bars, result['all_probabilities'] * 100)):
            axes[1].text(prob + 1, bar.get_y() + bar.get_height()/2,
                        f'{prob:.1f}%', va='center')

        # Highlight predicted class
        bars[result['class_index']].set_edgecolor('black')
        bars[result['class_index']].set_linewidth(2)

        plt.suptitle(f'Diagnosis: {result["predicted_class"]} '
                    f'({result["confidence"]:.1f}% confidence)',
                    fontsize=16, y=1.02)
        plt.tight_layout()
        plt.show()

        # Print summary
        print("\n" + "="*60)
        print("PREDICTION SUMMARY")
        print("="*60)
        print(f"Predicted Class: {result['predicted_class']}")
        print(f"Confidence: {result['confidence']:.1f}%")
        print("\nAll Probabilities:")
        for i, cls in enumerate(self.class_names):
            print(f"  {cls:15s}: {result['all_probabilities'][i]*100:6.2f}%")
        print("="*60)

# Initialize detector
detector = BellPepperDiseaseDetector(class_names=class_names)
print("Bell Pepper Disease Detector initialized successfully!")

Interactive Dashboard with HTML Interface

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import io
from PIL import Image
import numpy as np
import cv2
import base64 # Import the base64 module

# --- Widgets for the interface ---
# File upload widget
upload_widget = widgets.FileUpload(
    accept='.jpg,.jpeg,.png',
    multiple=False,
    description='Upload Image',
    button_style='info', # Style the button
    layout=widgets.Layout(width='auto', margin='10px auto') # Make it visible and centered
)

# Buttons
predict_button = widgets.Button(description="Analyze Image", button_style='success', layout=widgets.Layout(width='auto'))
clear_button = widgets.Button(description="Clear", button_style='warning', layout=widgets.Layout(width='auto'))

# Output widget for displaying status and results (this will replace 'output' from the original)
status_output = widgets.Output()

# HTML component for the main layout and display elements that don't need direct widget interaction
html_content = widgets.HTML(
    value="""
    <div style="font-family: Arial, sans-serif; max-width: 1200px; margin: 0 auto;">
        <div style="background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
                    padding: 20px; border-radius: 10px; color: white; text-align: center;">
            <h1 style="margin: 0; font-size: 28px;">🌱 Bell Pepper Leaf Disease Detection System</h1>
            <p style="font-size: 16px; opacity: 0.9;">Upload a bell pepper leaf image for automated disease diagnosis</p>
        </div>

        <div style="margin-top: 20px; display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">
            <div style="background: #f8f9fa; padding: 20px; border-radius: 8px; border: 1px solid #dee2e6;">
                <h3 style="color: #2c3e50; margin-top: 0;">📤 Upload Image</h3>
                <p style="color: #666; font-size: 14px;">Supported formats: JPG, JPEG, PNG</p>
                <!-- Placeholder for the actual image preview -->
                <div id="image-preview" style="display: none; text-align: center; margin-top: 15px;">
                    <img id="preview-img" style="max-width: 100%; max-height: 300px; border-radius: 5px; border: 1px solid #ddd;">
                    <p id="file-name" style="margin-top: 10px; font-weight: bold; color: #333;"></p>
                </div>
                <!-- The ipywidgets.FileUpload widget and buttons will be inserted below this HTML block by the VBox/HBox -->
            </div>

            <div style="background: #f8f9fa; padding: 20px; border-radius: 8px; border: 1px solid #dee2e6;">
                <h3 style="color: #2c3e50; margin-top: 0;">📊 Results</h3>
                <!-- The status and results will be displayed here via status_output -->
                <div id="status-message" style="padding: 15px; background: #e9ecef; border-radius: 5px; margin-bottom: 20px;">
                    <p style="margin: 0; color: #6c757d;">Status: Ready for image upload</p>
                </div>

                <div id="results-container" style="display: none;">
                    <h4 style="color: #495057; border-bottom: 2px solid #dee2e6; padding-bottom: 10px;">Diagnosis Results</h4>
                    <div id="results-content"></div>
                </div>

                <div id="instructions" style="margin-top: 20px; padding: 15px; background: #fff3cd;
                                              border-radius: 5px; border-left: 4px solid #ffc107;">
                    <h4 style="margin-top: 0; color: #856404;">ℹ️ Instructions</h4>
                    <ol style="margin: 10px 0; padding-left: 20px; color: #856404;">
                        <li>Use the "Upload Image" button below to select a bell pepper leaf image.</li>
                        <li>Click "Analyze Image" to process.</li>
                        <li>Review the disease diagnosis results.</li>
                        <li>Use "Clear" to reset and try another image.</li>
                    </ol>
                </div>
            </div>
        </div>

        <div style="margin-top: 20px; background: #f8f9fa; padding: 15px; border-radius: 8px;
                    border: 1px solid #dee2e6; font-size: 14px; color: #6c757d;">
            <p style="margin: 0;">⚠️ Note: This is an automated system. For critical decisions, consult with agricultural experts.</p>
        </div>
    </div>

    <style>
        #predict-button:hover { background: #218838; } /* These are no longer HTML buttons with these IDs but ipywidgets buttons */
        #clear-button:hover { background: #5a6268; } /* These are no longer HTML buttons with these IDs but ipywidgets buttons */
        .result-item { margin: 10px 0; padding: 10px; border-radius: 5px; }
        .healthy { background: #d4edda; color: #155724; border-left: 4px solid #28a745; }
        .diseased { background: #f8d7da; color: #721c24; border-left: 4px solid #dc3545; }
    </style>
    """
)

# Assemble the dashboard components
dashboard_container = widgets.VBox([
    html_content,
    widgets.HBox([upload_widget, predict_button, clear_button], layout=widgets.Layout(justify_content='center', margin='20px 0')),
    status_output # For Python print statements and general status
])

# Display the main dashboard container
display(dashboard_container)

# --- Python callback functions ---
# These now interact with the ipywidgets directly and update the HTML using JavaScript from Python.

def update_html_status(message, color='6c757d', background='e9ecef'):
    """Helper to update the status message in the HTML part."""
    js = f"""
    document.getElementById('status-message').innerHTML = '<p style="margin: 0; color: #{color};">Status: {message}</p>';
    document.getElementById('status-message').style.background = '#{background}';
    """
    with status_output:
        display(HTML(f'<script>{js}</script>'))

def on_upload_change(change):
    """Handle file upload via ipywidgets.FileUpload."""
    if upload_widget.value:
        uploaded_file = list(upload_widget.value.values())[0]
        file_name = uploaded_file['metadata']['name']

        # Encode content to base64 for embedding in HTML
        file_content_b64 = base64.b64encode(uploaded_file['content']).decode('ascii')

        # Update HTML preview via JavaScript
        js = f"""
        document.getElementById('preview-img').src = "data:image/png;base64,{file_content_b64}";
        document.getElementById('file-name').textContent = "{file_name}";
        document.getElementById('image-preview').style.display = 'block';
        document.getElementById('instructions').style.display = 'none'; // Hide instructions once image is uploaded
        updateStatus('Image uploaded successfully. Ready for analysis.', '17a2b8');
        """
        with status_output:
            display(HTML(f'<script>{js}</script>'))

def on_predict_click(b):
    """Handle predict button click"""
    with status_output:
        clear_output() # Clear previous output within the status_output area

        if not upload_widget.value:
            update_html_status('Please upload an image first.', 'dc3545')
            print("Please upload an image first.")
            return

        try:
            update_html_status('Processing image... Analyzing leaf health.', '17a2b8')

            # Get uploaded file
            uploaded_file = list(upload_widget.value.values())[0]
            file_name = uploaded_file['metadata']['name']
            file_content = uploaded_file['content']

            print(f"Processing: {file_name}")

            # Convert to PIL Image
            image = Image.open(io.BytesIO(file_content))

            # Make prediction
            result = detector.predict(image)

            if result:
                # Display results in matplotlib
                detector.display_results(result)

                # Update HTML results
                pred_class = result['predicted_class']
                confidence = result['confidence']
                all_probs = result['all_probabilities']

                # Create HTML for results
                results_html = f"""
                <div style="margin: 20px 0;">
                    <div class="result-item {'healthy' if 'healthy' in pred_class.lower() else 'diseased'}"
                         style="padding: 15px; border-radius: 5px; margin-bottom: 15px;">
                        <h3 style="margin: 0 0 10px 0;">Diagnosis: {pred_class}</h3>
                        <p style="margin: 0; font-size: 18px; font-weight: bold;">
                            Confidence: {confidence:.1f}%
                        </p>
                    </div>

                    <h4>Detailed Probabilities:</h4>
                    <table style="width: 100%; border-collapse: collapse; margin-top: 10px;">
                """

                for i, cls in enumerate(detector.class_names):
                    prob = all_probs[i] * 100
                    color = "#28a745" if 'healthy' in cls.lower() else "#dc3545"
                    results_html += f"""
                    <tr style="border-bottom: 1px solid #dee2e6;">
                        <td style="padding: 8px; width: 60%;">{cls}</td>
                        <td style="padding: 8px; width: 40%;">
                            <div style="background: #e9ecef; border-radius: 3px; height: 20px;">
                                <div style="background: {color}; width: {prob}%; height: 100%; border-radius: 3px;"></div>
                            </div>
                        </td>
                        <td style="padding: 8px; text-align: right; font-weight: bold;">{prob:.1f}%</td>
                    </tr>
                    """

                results_html += """
                    </table>

                    <div style="margin-top: 20px; padding: 15px; background: #e7f3ff; border-radius: 5px;">
                        <h4 style="margin-top: 0; color: #004085;">Recommendation:</h4>
                        <p style="margin: 0;">
                """

                if 'healthy' in pred_class.lower() and confidence > 70:
                    results_html += "✅ Leaf appears healthy. Continue regular monitoring and maintenance."
                elif 'healthy' in pred_class.lower():
                    results_html += "⚠️ Leaf appears mostly healthy. Monitor for any changes."
                elif 'diseased' in pred_class.lower() and confidence > 70:
                    results_html += "🚨 Disease detected with high confidence. Consider treatment options and isolate affected plants."
                else:
                    results_html += "⚠️ Possible disease detected. Monitor closely and consider expert consultation."

                results_html += """
                        </p>
                    </div>
                </div>
                """

                # Update HTML
                js_update_results = f"""
                document.getElementById('results-content').innerHTML = `{results_html}`;
                document.getElementById('results-container').style.display = 'block';
                updateStatus('Analysis complete. Results displayed below.', '28a745');
                """
                with status_output:
                    display(HTML(f'<script>{js_update_results}</script>'))

                print(f"Analysis complete: {pred_class} ({confidence:.1f}% confidence)")
            else:
                update_html_status('Error during analysis. Please try another image.', 'dc3545')
                print("Error during analysis. Please try another image.")

        except Exception as e:
            update_html_status(f'Error: {str(e)}', 'dc3545')
            print(f"Error: {e}")

def on_clear_click(b):
    """Handle clear button click"""
    upload_widget.value.clear() # Clear the FileUpload widget's value
    with status_output:
        clear_output() # Clear any previous Python output

    js = """
    document.getElementById('preview-img').src = '';
    document.getElementById('file-name').textContent = '';
    document.getElementById('image-preview').style.display = 'none';
    document.getElementById('instructions').style.display = 'block'; // Show instructions again
    document.getElementById('results-container').style.display = 'none';
    document.getElementById('results-content').innerHTML = '';
    updateStatus('Ready for image upload.', '6c757d');
    """
    with status_output:
        display(HTML(f'<script>{js}</script>'))
    print("Dashboard cleared. Ready for new image.")

# Attach event handlers
upload_widget.observe(on_upload_change, names='value')
predict_button.on_click(on_predict_click)
clear_button.on_click(on_clear_click)

# Helper JavaScript function to update status within the HTML, callable from Python
with status_output:
    display(HTML("""
    <script>
        function updateStatus(message, color, background = 'e9ecef') {
            const statusDiv = document.getElementById('status-message');
            if (statusDiv) {
                statusDiv.innerHTML = `<p style="margin: 0; color: #${color};">Status: ${message}</p>`;
                statusDiv.style.background = `#${background}`;
            }
        }
    </script>
    """
))

print("\nInteractive dashboard loaded successfully!")
print("Upload a bell pepper leaf image using the interface above.")

Test with Sample Images from Dataset

In [ ]:
def test_with_dataset_samples():
    """Test the model with actual images from the dataset"""

    print("Testing model with actual dataset samples...")
    print("=" * 60)

    # Get sample images from each class
    for cls in class_names:
        cls_path = os.path.join(DATASET_PATH, cls)
        if os.path.exists(cls_path):
            image_files = [f for f in os.listdir(cls_path)
                          if f.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp'))]

            if image_files:
                # Take first image from each class
                test_image = image_files[0]
                test_path = os.path.join(cls_path, test_image)

                print(f"\nTesting with {cls} image: {test_image}")
                print("-" * 40)

                try:
                    # Load and predict
                    image = Image.open(test_path)
                    result = detector.predict(image)

                    if result:
                        # Display results
                        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

                        # Show image
                        axes[0].imshow(result['original_image'])
                        axes[0].set_title(f'Actual: {cls}\nFile: {test_image}', fontsize=12)
                        axes[0].axis('off')

                        # Show probabilities
                        x_pos = np.arange(len(class_names))
                        colors = ['green' if 'healthy' in c.lower() else 'red' for c in class_names]

                        bars = axes[1].barh(x_pos, result['all_probabilities'] * 100, color=colors)
                        axes[1].set_yticks(x_pos)
                        axes[1].set_yticklabels(class_names)
                        axes[1].set_xlabel('Confidence (%)')
                        axes[1].set_xlim([0, 100])
                        axes[1].set_title(f'Predicted: {result["predicted_class"]}\nConfidence: {result["confidence"]:.1f}%', fontsize=12)

                        # Add values
                        for i, (bar, prob) in enumerate(zip(bars, result['all_probabilities'] * 100)):
                            axes[1].text(prob + 1, bar.get_y() + bar.get_height()/2,
                                       f'{prob:.1f}%', va='center')

                        plt.tight_layout()
                        plt.show()

                        # Check if prediction matches actual
                        if result['predicted_class'].lower() == cls.lower():
                            print(f"✓ CORRECT: Predicted '{result['predicted_class']}' (Confidence: {result['confidence']:.1f}%)")
                        else:
                            print(f"✗ MISMATCH: Predicted '{result['predicted_class']}' but actual is '{cls}'")

                except Exception as e:
                    print(f"Error testing {test_image}: {e}")

    print("\n" + "=" * 60)
    print("Sample testing completed.")
    print("Use the interactive dashboard above to test your own images.")

# Run sample tests (optional)
# test_with_dataset_samples()